# C2 · Extracción por apertura

**Spec:** [`docs/spec_C2_codex_aperture_extraction.md`](../docs/spec_C2_codex_aperture_extraction.md)  |  **Bloque:** C · Extracción  |  **Run de este set:** `ROXs42Bb_realigned`

Extrae el espectro del compañero por apertura, con controles al mismo radio.

| | |
|---|---|
| **Entrada** | Cubo + posición |
| **Salida (QC/productos)** | `stages/spec_aperture_qc.json` |
| **Consume aguas abajo** | D1, E1 (controles) |


## Qué hace C2 y por qué

C2 extrae el espectro del compañero por **apertura** (`box3` por defecto, `box5`) y define el **contrato de producto espectral estándar** (`wave, flux, flux_err, apcorr, npix, flags`) sobre el que se construyen **todos** los extractores (C3, C4) y D1/E1. Es infraestructura: cero ciencia nueva, la extracción ya estaba validada.

**Decisiones clave:**
- **`control = objeto`:** 33 aperturas de control se procesan **idénticas** al objeto (mismo radio, annulus de fondo, apcorr) → el σ es **empírico** (M5 STAT rojo, así que no se usa el STAT directo). [`docs/noise_model.md`](../docs/noise_model.md)
- **Corrección de apertura (growth-curve de la PSF):** box3 capta solo una fracción de la **PSF AO ancha** → una corrección grande y **cromática** (mediana ~44.7×, ~118× en el azul → ~21× en el rojo). El chequeo `v4_apcorr_range` **falla** por ese rango enorme — se marca, no se esconde.
- **Apcorr con extracción *wings-intact*** (cubo crudo + annulus), **no** el residual de 04b: 04b sobre-sustrae las alas del compañero (daría box5 < box3, no físico), así que la curva de crecimiento se mantiene auto-consistente sobre el cubo crudo.

La apertura NO es el método canónico (G1 la **rechaza** por insensible en el borde del compañero); C2 aporta el contrato de producto y el control 'A: apertura' para D1.


## Cómo ejecutar de forma independiente

```bash
conda activate MUSE               # kernel/env con astropy + musepipe
export RUN=ROXs42Bb_realigned   # el run de este objeto
cd MUSE-accretion-pipeline                    # raíz del repo
bash scripts/stage_x01_aperture.sh --run-id $RUN
```

Ligero–moderado.

La celda de abajo hace lo mismo desde el notebook (guardada por `RUN`).


In [ ]:
import os, sys
# Localiza la raíz del repo ascendiendo hasta encontrar `musepipe/` (robusto a
# la profundidad: funciona con el cwd en notebooks/<obj>/, en notebooks/ o en la
# raíz). Añade la raíz (para `import musepipe`) y notebooks/ (para `_nbcommon`).
_d = os.getcwd()
while _d != os.path.dirname(_d):
    if os.path.isdir(os.path.join(_d, 'musepipe')) and os.path.isdir(os.path.join(_d, 'notebooks')):
        break
    _d = os.path.dirname(_d)
_root = _d
for _p in (_root, os.path.join(_root, 'notebooks')):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
RUN_ID = nb.resolve_run_id('ROXs42Bb_realigned')
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))
print('QC   =', nb.provenance_line('stages/spec_aperture_qc.json', RUN_ID))


## Ejecutar o auditar


In [ ]:
RUN = False   # -> True para RE-EJECUTAR esta etapa (regenera su QC)

if RUN:
    cmd = 'bash scripts/stage_x01_aperture.sh --run-id $RUN'.replace('$RUN', RUN_ID)
    print('ejecutando:', cmd)
    import subprocess
    subprocess.run(cmd, shell=True, cwd=str(nb.project_root()), check=True)
else:
    print('Modo auditoría (RUN=False): se carga el QC existente abajo.')


## QC / resultados


In [ ]:
qc = nb.load_qc_optional('stages/spec_aperture_qc.json', RUN_ID)
nb.show(qc, keys=['apertures', 'errors.mode', 'aperture_correction.median', 'v4_apcorr_range_ok', 'bad_window_channels'], title='C2')


## Resultados que llevaron a la conclusión

Aperturas, errores empíricos, corrección de apertura y chequeos del `spec_aperture_qc.json`.


In [ ]:
if qc is None:
    print('(evidencia omitida: la etapa no se ha ejecutado para esta cadena)')
else:
    with nb.evidence_guard('C2', 'stages/spec_aperture_qc.json'):
        q = nb.load_qc('stages/spec_aperture_qc.json', RUN_ID)
        err = q['errors']; ac = q['aperture_correction']; ck = q['checks']
        print('apertures:', q['apertures'], '| posiciones de:', q['positions_from'].split('/')[-1])
        print(f"errores: modo={err['mode']}, covarianza box3={err['covariance_factor_box3']:.2f}×, "
              f"stat/empírico={err['stat_vs_empirical_median_ratio']:.2f}")
        print(f"apcorr ({ac['mode']}): mediana {ac['median']:.1f}× , máx {ac['max']:.1f}× , norm r={ac['norm_radius_px']:.0f}px")
        print(f"flags: bad-window {q['flags']['bad_window_channels']}, skyline {q['flags']['skyline_channels']}, clipped {q['flags']['clipped_channels']}")
        print(f"checks: v2_error_ratio={ck['v2_error_ratio_ok']} v3_roundtrip={ck['v3_roundtrip_ok']} v4_apcorr_range={ck['v4_apcorr_range_ok']}  (v4 falla: apcorr enorme)")
        print()
        for i, s in enumerate(q['open_issues'], 1):
            print(f'  open_issue {i}: {s}')


## Plot 1 — el espectro por apertura (box3) + ruido empírico

Del producto `spec_aperture_object.fits` y los 33 controles (`spec_aperture_controls.npz`). El flujo es **muy ruidoso** (banda ±1σ empírica); el continuo suavizado **sube al rojo** (SED real de enana fría) y **no hay nada en Hα** (contexto de la no-detección). El hueco es la ventana del láser AO.


In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
    from astropy.io import fits
    rd = nb.run_dir(RUN_ID)
    h = fits.open(rd / 'stages' / 'spec_aperture_object.fits'); d = h[1].data
    w = np.asarray(d['wave_A'], float); flux = np.asarray(d['flux'], float); h.close()
    ctrl = np.load(rd / 'stages' / 'spec_aperture_controls.npz')['control_spectra']
    sig = np.nanstd(ctrl, axis=0)
    k = np.ones(41) / 41; sm = np.convolve(np.nan_to_num(flux), k, mode='same')
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.fill_between(w, -sig, sig, color='0.8', label=f'±1σ empírico ({ctrl.shape[0]} controles)')
    ax.plot(w, flux, lw=0.3, color='0.5', alpha=0.6)
    ax.plot(w, sm, lw=1.2, color='tab:blue', label='flujo compañero (suavizado 41ch)')
    ax.axvline(6563, color='tab:red', ls=':', label='Hα')
    ax.set_ylim(np.nanpercentile(flux, 2), np.nanpercentile(flux, 98))
    ax.set_xlabel('λ [Å]'); ax.set_ylabel('flujo (apcorr aplicada)')
    ax.set_title('C2 · espectro por apertura box3 del compañero (errores empíricos)')
    ax.legend(fontsize=8); fig.tight_layout()
    outdir = rd / 'plots' / 'c2_aperture'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'spectrum.png', dpi=110); print('figura ->', outdir / 'spectrum.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Plot 2 — la corrección de apertura cromática

La corrección `box3 → flujo total` baja de ~118× en el azul (PSF AO peor) a ~21× en el rojo (PSF más apretada); mediana ~44.7×. Su rango enorme es lo que hace fallar `v4_apcorr_range` — es real (PSF AO ancha), no un defecto.


In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
    from astropy.io import fits
    rd = nb.run_dir(RUN_ID)
    h = fits.open(rd / 'stages' / 'spec_aperture_object.fits'); d = h[1].data
    w = np.asarray(d['wave_A'], float); apc = np.asarray(d['apcorr'], float); h.close()
    fig, ax = plt.subplots(figsize=(9, 3.6))
    ax.plot(w, apc, lw=0.8, color='tab:purple')
    ax.axhline(np.nanmedian(apc), color='k', ls='--', lw=1, label=f'mediana {np.nanmedian(apc):.1f}×')
    ax.set_xlabel('λ [Å]'); ax.set_ylabel('corrección de apertura ×')
    ax.set_title('C2 · corrección de apertura (box3 capta poco de la PSF AO ancha)')
    ax.legend(fontsize=8); fig.tight_layout()
    outdir = rd / 'plots' / 'c2_aperture'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'apcorr.png', dpi=110); print('figura ->', outdir / 'apcorr.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Decisiones y notas
- Principio **control = objeto**: 33 controles con annulus bkg + apcorr, procesados idénticos al objeto → σ **empírico** (M5 rojo). · [`docs/noise_model.md`](../docs/noise_model.md)
- **Corrección de apertura cromática grande** (mediana 44.7×): box3 capta poco de la PSF AO ancha; `v4_apcorr_range` falla por el rango (marcado, no oculto).
- **Apcorr wings-intact** (cubo crudo + annulus), no el residual 04b, que sobre-sustrae las alas (box5<box3).
- Apertura **no canónica**: G1 la rechaza por insensible en el borde; C2 aporta el contrato de producto y el control 'A' para D1.


## Conclusión (registrada)

**C2: espectro por apertura (box3/box5) en el formato de producto estándar; errores empíricos; apcorr cromática mediana 44.7×.**

- **Fecha:** cadena D1 v2 sobre el run realineado (2026-07-09).
- **Entrada:** cubo stage02; posiciones de B3; 33 controles.
- **Espectro:** muy ruidoso, continuo real que sube al rojo (enana fría), **nada en Hα** (no-detección).
- **Errores empíricos** (M5 STAT rojo); covarianza box3 = 6.38×.
- **Apcorr:** growth-curve, 118× (azul) → 21× (rojo); `v4_apcorr_range` falla por el rango (real, PSF AO ancha); apcorr wings-intact para no sobre-sustraer alas.
- **Rol:** contrato de producto para C3/C4/D1/E1; método de apertura no canónico (psffit lo es).
